# 04 - XGBoost Regressor

> Gradient Boosted Trees — a powerful supervised ML model using engineered lag features, rolling statistics, and calendar indicators to predict monthly sales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
COLORS = {'primary':'#003B73','secondary':'#0074B7','alert':'#BF212F','success':'#27AE60'}
FEATURES = ['month','quarter','year','trend','lag_1','lag_2','lag_3','lag_12','rolling_mean_3','rolling_mean_6','rolling_std_3']
print("Libraries loaded")

## 1. Load Prepared ML Data

In [ ]:
train = pd.read_csv('../data/train_ml.csv', parse_dates=['ds'])
test = pd.read_csv('../data/test_ml.csv', parse_dates=['ds'])
X_train, y_train = train[FEATURES], train['y']
X_test, y_test = test[FEATURES], test['y']
print(f"Train: {len(train)} months | Test: {len(test)} months")
print(f"Features: {len(FEATURES)}")

## 2. Train XGBoost

In [ ]:
model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, subsample=0.8,
                     colsample_bytree=0.8, random_state=42, reg_alpha=0.1, reg_lambda=1.0)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print("XGBoost model trained")

## 3. Evaluate

In [ ]:
y_true = y_test.values
y_pred = model.predict(X_test)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
print(f"XGBoost Test Metrics:")
print(f"  MAPE:  {mape:.2f}%")
print(f"  MAE:   ${mae:,.2f}")
print(f"  RMSE:  ${rmse:,.2f}")
print(f"  R2:    {r2:.4f}")

## 4. Feature Importance

In [ ]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
plt.figure(figsize=(10,6))
imp.plot.barh(color=COLORS['secondary'], edgecolor='white')
plt.title('XGBoost Feature Importance', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../visualizations/xgboost_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Visualize Predictions

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(test['ds'], y_true, label='Actual (Test)', color='black', linewidth=2, marker='o')
plt.plot(test['ds'], y_pred, label=f'XGBoost Pred (MAPE={mape:.1f}%)', color=COLORS['secondary'], linestyle='--', linewidth=2, marker='x')
plt.title('XGBoost Forecast vs Actuals', fontsize=15, fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Monthly Sales ($)'); plt.legend()
plt.tight_layout()
plt.savefig('../visualizations/xgboost_forecast.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Save Model & Predictions

In [ ]:
pred_df = pd.DataFrame({'ds': test['ds'].values, 'actual': y_true, 'predicted': y_pred})
pred_df.to_csv('../predictions/xgboost_predictions.csv', index=False)
joblib.dump(model, '../models/xgboost_model.pkl')
print("Saved: predictions/xgboost_predictions.csv")
print("Saved: models/xgboost_model.pkl")
print("\nProceed to 05_Random_Forest_Model.ipynb")